### Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

### Read from bronze table

In [0]:
df = spark.table("workspace.bronze.crm_cust_info_raw")

df.display()

In [0]:
df.printSchema()

### Silver transformations

Rename columns with proper column names

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

Trim whitespaces in the data

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

Replace short-form data with proper column names

In [0]:
df = (
    df
    .withColumn(
        "marital_status",
        F.when(F.upper(F.col("marital_status")) == "S", "Single")
         .when(F.upper(F.col("marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .withColumn(
        "gender",
        F.when(F.upper(F.col("gender")) == "F", "Female")
         .when(F.upper(F.col("gender")) == "M", "Male")
         .otherwise("n/a")
    )
)

Remove Records with Missing Customer ID

In [0]:
# Count nulls in 'customer_id'
df.filter(col("customer_id").isNull()).count()

In [0]:
df = df.filter(col("customer_id").isNotNull())

### Write into Silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")

In [0]:
%sql
SELECT * 
FROM workspace.silver.crm_customers 
LIMIT 10